# Leakage-free federated deep learning for memory-based malware family attribution (CIC-MalMem-2022)
**Kaggle setup:** Accelerator **GPU T4 ×1**, Internet **On** (only used to `pip install opacus` for the ε accountant).
**Input dataset:** https://www.kaggle.com/datasets/luccagodoy/obfuscated-malware-memory-2022-cic
(any copy of CIC-MalMem-2022 works; the loader finds the CSV automatically).

Experiments (all outputs → `/kaggle/working/malmem_results/`):
* **E1** Leakage: random record split vs split grouped by malware sample hash (binary / category / family).
* **E2** Deep learning vs trees under the grouped split (MLP, FT-Transformer, 1D-CNN, RF, XGBoost).
* **E3** Federated learning: centralized vs local-only vs FedAvg vs FedProx; IID and family-skewed non-IID clients.
* **E4** Privacy: DP-SGD inside FedAvg, privacy (ε) vs family macro-F1.
* **E5** Unseen-family recognition: leave-one-family-out, 15 families, 8 uncertainty scores (+ FedAvg model).

In [ ]:
import os, sys, glob, json, time, math, copy, warnings, subprocess
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as Fn
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import NearestNeighbors
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FAST = bool(int(os.environ.get("MALMEM_FAST", "0")))           # quick smoke test
CFG = dict(DATA=os.environ.get("MALMEM_DATA", "/kaggle/input"), OUT=os.environ.get("MALMEM_OUT", "/kaggle/working/malmem_results"),
           SEEDS=[0, 1, 2], EPOCHS=60, PAT=8, BS=512, LR=1e-3,
           CLIENTS=10, ROUNDS=60, LOCAL_EP=2, MU=0.01, DIR_ALPHAS=[0.5, 0.1],
           DP_SIGMAS=[0.6, 1.0, 1.5], DP_CLIP=1.0, DP_DELTA=1e-5, LOFO_ROUNDS=30, MC_T=20, KNN_K=50)
if FAST: CFG.update(SEEDS=[0], EPOCHS=3, PAT=2, ROUNDS=3, LOCAL_EP=1, DP_SIGMAS=[1.0], LOFO_ROUNDS=2, MC_T=3, DIR_ALPHAS=[0.5])
os.makedirs(CFG["OUT"], exist_ok=True); R = {}
print("device", DEV, torch.cuda.get_device_name(0) if DEV.type == "cuda" else "", "| FAST", FAST)
def seed_all(s): np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
def save(): json.dump(R, open(f"{CFG['OUT']}/results.json", "w"), indent=1)

# privacy accountant (Opacus RDP); install if missing
try:
    from opacus.accountants import RDPAccountant
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "opacus"], check=False)
    try: from opacus.accountants import RDPAccountant
    except Exception: RDPAccountant = None
print("RDP accountant available:", RDPAccountant is not None)

## Data

In [ ]:
def find_csv():
    for f in glob.glob(os.path.join(CFG["DATA"], "**", "*.csv"), recursive=True):
        try:
            if "Class" in pd.read_csv(f, nrows=1).columns: return f
        except Exception: pass
    raise FileNotFoundError("CIC-MalMem-2022 CSV not found under " + CFG["DATA"])
d = pd.read_csv(find_csv())
src = d["Filename"].astype(str) if "Filename" in d.columns else d["Category"].astype(str)   # e.g. Ransomware-Ako-<sha256>-1.raw
mal = d["Class"].astype(str).str.strip().eq("Malware")
parts = src.str.split("-")
d["cat"] = np.where(mal, parts.str[0], "Benign")
d["fam"] = np.where(mal, parts.str[1], "Benign")
d["grp"] = np.where(mal, parts.str[2], "benign_" + d.index.astype(str))   # benign dumps have no sample id
d["bin"] = np.where(mal, "Malware", "Benign")
FEAT = [c for c in d.columns if c not in ("Class", "Category", "Filename", "cat", "fam", "grp", "bin")
        and pd.api.types.is_numeric_dtype(d[c]) and d[c].nunique() > 1]
FAMS = sorted(f for f in d.fam.unique() if f != "Benign")
R["N_ROWS"] = int(len(d)); R["N_FEAT"] = len(FEAT); R["N_FAMILIES"] = len(FAMS)
R["N_MAL_SAMPLES"] = int(d.loc[mal, "grp"].nunique()); R["DUMPS_PER_SAMPLE"] = round(float(d.loc[mal].groupby("grp").size().mean()), 2)
print(f"rows {len(d)}, features {len(FEAT)}, families {len(FAMS)}, malware samples {R['N_MAL_SAMPLES']}, dumps/sample {R['DUMPS_PER_SAMPLE']}")

def split(kind, seed, test=0.2, val=0.2):
    idx = np.arange(len(d))
    if kind == "random":
        tr, te = train_test_split(idx, test_size=test, random_state=seed, stratify=d.fam)
        tr, va = train_test_split(tr, test_size=val / (1 - test), random_state=seed, stratify=d.fam.values[tr])
    else:
        g = d.grp.values
        tr, te = next(GroupShuffleSplit(1, test_size=test, random_state=seed).split(idx, groups=g))
        a, b = next(GroupShuffleSplit(1, test_size=val / (1 - test), random_state=seed).split(tr, groups=g[tr])); tr, va = tr[a], tr[b]
    return tr, va, te

class Scaler:
    def fit(self, X): z = np.sign(X) * np.log1p(np.abs(X)); self.m, self.s = z.mean(0), z.std(0) + 1e-6; return self
    def __call__(self, X): z = np.sign(X) * np.log1p(np.abs(X)); return ((z - self.m) / self.s).astype(np.float32)
XALL = d[FEAT].values.astype(np.float64)
def arrays(task, tr, va, te, classes=None):
    classes = classes or sorted(d[task].unique()); cl = {c: i for i, c in enumerate(classes)}
    sc = Scaler().fit(XALL[tr]); y = d[task].map(cl).values
    return sc(XALL[tr]), y[tr], sc(XALL[va]), y[va], sc(XALL[te]), y[te], classes

## Models

In [ ]:
class MLP(nn.Module):
    def __init__(s, d_in, C, w=256, drop=0.2, norm="bn"):
        super().__init__(); N = (lambda k: nn.BatchNorm1d(k)) if norm == "bn" else (lambda k: nn.LayerNorm(k))
        s.body = nn.Sequential(nn.Linear(d_in, w), N(w), nn.ReLU(), nn.Dropout(drop),
                               nn.Linear(w, w // 2), N(w // 2), nn.ReLU(), nn.Dropout(drop),
                               nn.Linear(w // 2, w // 4), N(w // 4), nn.ReLU(), nn.Dropout(drop))
        s.head = nn.Linear(w // 4, C)
    def forward(s, x): return s.head(s.body(x))

class FTTransformer(nn.Module):
    """Feature-tokenizer transformer (Gorishniy et al., 2021), compact."""
    def __init__(s, d_in, C, dm=64, L=3, H=4, drop=0.1):
        super().__init__(); s.W = nn.Parameter(torch.randn(d_in, dm) * 0.02); s.b = nn.Parameter(torch.zeros(d_in, dm))
        s.cls = nn.Parameter(torch.zeros(1, 1, dm))
        s.enc = nn.TransformerEncoder(nn.TransformerEncoderLayer(dm, H, dm * 2, drop, batch_first=True, norm_first=True), L)
        s.body = nn.Identity(); s.head = nn.Sequential(nn.LayerNorm(dm), nn.ReLU(), nn.Linear(dm, C))
    def emb(s, x):
        t = x.unsqueeze(-1) * s.W + s.b; t = torch.cat([s.cls.expand(len(x), -1, -1), t], 1); return s.enc(t)[:, 0]
    def forward(s, x): return s.head(s.emb(x))

class CNN1D(nn.Module):
    def __init__(s, d_in, C, ch=64, drop=0.2):
        super().__init__()
        s.conv = nn.Sequential(nn.Conv1d(1, ch, 3, padding=1), nn.BatchNorm1d(ch), nn.ReLU(),
                               nn.Conv1d(ch, ch, 3, padding=1), nn.BatchNorm1d(ch), nn.ReLU(), nn.AdaptiveAvgPool1d(8))
        s.head = nn.Sequential(nn.Flatten(), nn.Dropout(drop), nn.Linear(ch * 8, 128), nn.ReLU(), nn.Linear(128, C))
    def forward(s, x): return s.head(s.conv(x.unsqueeze(1)))

def T(a, dt=torch.float32): return torch.as_tensor(a, dtype=dt, device=DEV)
def cweights(y, C):
    c = np.bincount(y, minlength=C).astype(float); c[c == 0] = 1; w = len(y) / (C * c); return T(w / w.mean())

def edl_loss(lg, y, C, ep):
    a = Fn.softplus(lg) + 1; S = a.sum(1, keepdim=True); Y = Fn.one_hot(y, C).float(); p = a / S
    mse = ((Y - p) ** 2 + p * (1 - p) / (S + 1)).sum(1); at = a * (1 - Y) + Y; St = at.sum(1, keepdim=True)
    kl = torch.lgamma(St).squeeze(1) - math.lgamma(C) - torch.lgamma(at).sum(1) + ((at - 1) * (torch.digamma(at) - torch.digamma(St))).sum(1)
    return (mse + min(1.0, ep / 10) * kl).mean()

def loss_of(kind, out, y, C, w, ep):
    if kind == "edl": return edl_loss(out, y, C, ep)
    if kind == "ovr": return Fn.binary_cross_entropy_with_logits(out, Fn.one_hot(y, C).float(), weight=w.unsqueeze(0).expand(len(y), -1))
    return Fn.cross_entropy(out, y, weight=w)

def train(model, Xtr, ytr, Xva, yva, C, seed, kind="ce", epochs=None, init=None):
    seed_all(seed); net = model.to(DEV)
    if init is not None: net.load_state_dict(init)
    opt = torch.optim.AdamW(net.parameters(), CFG["LR"], weight_decay=1e-4); w = cweights(ytr, C)
    Xt, yt, Xv, yv = T(Xtr), T(ytr, torch.long), T(Xva), T(yva, torch.long); g = torch.Generator(device="cpu").manual_seed(seed)
    best, bad, state = 1e9, 0, None
    for ep in range(epochs or CFG["EPOCHS"]):
        net.train(); perm = torch.randperm(len(Xt), generator=g).to(DEV)
        for k in range(0, len(Xt), CFG["BS"]):
            j = perm[k:k + CFG["BS"]]
            if len(j) < 2: continue
            l = loss_of(kind, net(Xt[j]), yt[j], C, w, ep); opt.zero_grad(); l.backward(); opt.step()
        net.eval()
        with torch.no_grad(): vl = loss_of(kind, net(Xv), yv, C, w, ep).item()
        if vl < best - 1e-5: best, bad, state = vl, 0, copy.deepcopy(net.state_dict())
        else:
            bad += 1
            if bad >= CFG["PAT"]: break
    net.load_state_dict(state); net.eval(); return net

@torch.no_grad()
def logits(net, X, mc=False):
    net.eval()
    if mc:
        for m in net.modules():
            if isinstance(m, nn.Dropout): m.train()
    out = torch.cat([net(T(X[k:k + 8192])) for k in range(0, len(X), 8192)]).cpu().numpy(); net.eval(); return out
def sm(z): z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)
def ent(p): return -(p * np.log(p + 1e-12)).sum(1)
def scores(y, pred): return dict(acc=accuracy_score(y, pred), mf1=f1_score(y, pred, average="macro"))
def xgb_model(seed, C):
    import xgboost as xgb
    return xgb.XGBClassifier(n_estimators=400, max_depth=8, learning_rate=0.1, subsample=0.9, colsample_bytree=0.8,
                             tree_method="hist", device="cuda" if DEV.type == "cuda" else "cpu", random_state=seed,
                             objective="multi:softprob" if C > 2 else "binary:logistic")
def fmt(v, d=4): v = np.asarray(v, float); return f"{v.mean():.{d}f} ± {v.std(ddof=1):.{d}f}" if len(v) > 1 else f"{v.mean():.{d}f}"

## E1 — record-level leakage: random vs sample-grouped split

In [ ]:
t0 = time.time(); rows = []
for seed in CFG["SEEDS"]:
    for kind in ["random", "grouped"]:
        tr, va, te = split(kind, seed)
        for task in ["bin", "cat", "fam"]:
            Xa, ya, Xv, yv, Xt, yt, cls = arrays(task, tr, va, te); C = len(cls)
            rf = RandomForestClassifier(300, n_jobs=-1, random_state=seed).fit(Xa, ya); rows.append(dict(model="RF", split=kind, task=task, seed=seed, **scores(yt, rf.predict(Xt))))
            xg = xgb_model(seed, C).fit(Xa, ya); pr = xg.predict_proba(Xt); pr = pr.argmax(1) if pr.ndim == 2 else (pr > .5)
            rows.append(dict(model="XGBoost", split=kind, task=task, seed=seed, **scores(yt, pr)))
            net = train(MLP(Xa.shape[1], C), Xa, ya, Xv, yv, C, seed); rows.append(dict(model="MLP", split=kind, task=task, seed=seed, **scores(yt, logits(net, Xt).argmax(1))))
    print("E1 seed", seed, f"{time.time()-t0:.0f}s", flush=True)
E1 = pd.DataFrame(rows); E1.to_csv(f"{CFG['OUT']}/E1_leakage_raw.csv", index=False)
E1s = E1.groupby(["task", "model", "split"])[["acc", "mf1"]].agg(["mean", "std"]).round(4); E1s.to_csv(f"{CFG['OUT']}/E1_leakage.csv"); print(E1s)
for (task, model), g in E1.groupby(["task", "model"]):
    for sp in ["random", "grouped"]:
        R[f"E1_{task}_{model}_{sp}_MF1"] = fmt(g[g.split == sp].mf1); R[f"E1_{task}_{model}_{sp}_ACC"] = fmt(g[g.split == sp].acc)
    R[f"E1_{task}_{model}_DROP_MF1_PP"] = round(100 * (g[g.split == "random"].mf1.mean() - g[g.split == "grouped"].mf1.mean()), 2)
save()

## E2 — deep learning vs trees under the grouped split (category and family)

In [ ]:
rows, cms = [], {}
for seed in CFG["SEEDS"]:
    tr, va, te = split("grouped", seed)
    for task in ["cat", "fam"]:
        Xa, ya, Xv, yv, Xt, yt, cls = arrays(task, tr, va, te); C = len(cls)
        for name, mk in [("MLP", lambda: MLP(Xa.shape[1], C)), ("FT-Transformer", lambda: FTTransformer(Xa.shape[1], C)), ("1D-CNN", lambda: CNN1D(Xa.shape[1], C))]:
            t1 = time.time(); net = train(mk(), Xa, ya, Xv, yv, C, seed); p = logits(net, Xt).argmax(1)
            rows.append(dict(model=name, task=task, seed=seed, train_s=time.time() - t1, **scores(yt, p)))
            if task == "fam" and seed == CFG["SEEDS"][0]: cms[name] = (confusion_matrix(yt, p, labels=range(C)), cls)
        rf = RandomForestClassifier(300, n_jobs=-1, random_state=seed).fit(Xa, ya); p = rf.predict(Xt)
        rows.append(dict(model="RF", task=task, seed=seed, **scores(yt, p)))
        if task == "fam" and seed == CFG["SEEDS"][0]: cms["RF"] = (confusion_matrix(yt, p, labels=range(C)), cls)
        xg = xgb_model(seed, C).fit(Xa, ya); rows.append(dict(model="XGBoost", task=task, seed=seed, **scores(yt, xg.predict_proba(Xt).argmax(1))))
    print("E2 seed", seed, flush=True)
E2 = pd.DataFrame(rows); E2.to_csv(f"{CFG['OUT']}/E2_models_raw.csv", index=False)
E2s = E2.groupby(["task", "model"])[["acc", "mf1"]].agg(["mean", "std"]).round(4); E2s.to_csv(f"{CFG['OUT']}/E2_models.csv"); print(E2s)
for (task, model), g in E2.groupby(["task", "model"]): R[f"E2_{task}_{model}_MF1"] = fmt(g.mf1); R[f"E2_{task}_{model}_ACC"] = fmt(g.acc)
save()

## E3 — federated learning (MLP with LayerNorm so FL and DP use the same model)

In [ ]:
def partition(tr_idx, K, alpha, seed):
    """Assign whole malware samples (groups) to clients. alpha=None -> IID; else Dirichlet(alpha) over families."""
    rng = np.random.default_rng(seed); sub = d.iloc[tr_idx]; clients = [[] for _ in range(K)]
    for fam, gdf in sub.groupby("fam"):
        groups = gdf.grp.unique(); rng.shuffle(groups)
        if alpha is None or fam == "Benign": props = np.ones(K) / K
        else: props = rng.dirichlet(alpha * np.ones(K))
        cuts = (np.cumsum(props) * len(groups)).astype(int)[:-1]
        for k, gs in enumerate(np.split(groups, cuts)): clients[k].extend(gdf.index[gdf.grp.isin(gs)].tolist())
    pos = {v: i for i, v in enumerate(d.index)}
    return [np.array([pos[i] for i in c], dtype=int) for c in clients]

def per_sample_dp_step(net, xb, yb, w, opt, clip, sigma):
    """DP-SGD step: per-sample clipping (torch.func) + Gaussian noise."""
    from torch.func import functional_call, vmap, grad
    params = {k: v.detach() for k, v in net.named_parameters()}
    def loss_one(p, x, y): return Fn.cross_entropy(functional_call(net, p, (x.unsqueeze(0),)), y.unsqueeze(0), weight=w)
    g = vmap(grad(loss_one), in_dims=(None, 0, 0), randomness="different")(params, xb, yb)
    norms = torch.sqrt(sum((gi.reshape(len(xb), -1) ** 2).sum(1) for gi in g.values()))
    factor = (clip / (norms + 1e-6)).clamp(max=1.0)
    for (k, p_) in net.named_parameters():
        gs = (g[k] * factor.view(-1, *[1] * (g[k].dim() - 1))).sum(0)
        p_.grad = (gs + torch.randn_like(gs) * sigma * clip) / len(xb)
    opt.step(); opt.zero_grad()

def local_train(gnet, X, y, C, w, epochs, mu=0.0, dp=None, seed=0):
    net = copy.deepcopy(gnet); net.train(); opt = torch.optim.SGD(net.parameters(), lr=0.05, momentum=0.9 if dp is None else 0.0)
    g0 = [p.detach().clone() for p in gnet.parameters()]; Xt, yt = T(X), T(y, torch.long); steps = 0
    gen = torch.Generator(device="cpu").manual_seed(seed)
    for _ in range(epochs):
        perm = torch.randperm(len(Xt), generator=gen).to(DEV)
        for k in range(0, len(Xt), 128):
            j = perm[k:k + 128]
            if len(j) < 2: continue
            if dp is not None: per_sample_dp_step(net, Xt[j], yt[j], w, opt, CFG["DP_CLIP"], dp); steps += 1; continue
            l = Fn.cross_entropy(net(Xt[j]), yt[j], weight=w)
            if mu > 0: l = l + mu / 2 * sum(((p - q) ** 2).sum() for p, q in zip(net.parameters(), g0))
            opt.zero_grad(); l.backward(); opt.step(); steps += 1
    return net.state_dict(), len(X), steps

def fed_run(Xs, ys, Xva, yva, Xt, yt, C, rounds, mu=0.0, dp=None, seed=0, d_in=None, track=True):
    seed_all(seed); g = MLP(d_in, C, norm="ln").to(DEV); hist = []; w = cweights(np.concatenate(ys), C); steps_max = 0
    for r in range(rounds):
        states, ns = [], []
        for k, (X, y) in enumerate(zip(Xs, ys)):
            if len(X) == 0: continue
            s_, n_, st = local_train(g, X, y, C, w, CFG["LOCAL_EP"], mu, dp, seed * 1000 + r * 50 + k); states.append(s_); ns.append(n_); steps_max = max(steps_max, st)
        tot = sum(ns); g.load_state_dict({k: sum(s_[k] * (n_ / tot) for s_, n_ in zip(states, ns)) for k in states[0]})
        if track and (r % 5 == 4 or r == rounds - 1): hist.append((r + 1, f1_score(yt, logits(g, Xt).argmax(1), average="macro")))
    return g, hist, steps_max

rows, curves = [], {}
for seed in CFG["SEEDS"]:
    tr, va, te = split("grouped", seed); Xa, ya, Xv, yv, Xt, yt, cls = arrays("fam", tr, va, te); C = len(cls)
    sc = Scaler().fit(XALL[tr]); cl = {c: i for i, c in enumerate(cls)}
    net = train(MLP(Xa.shape[1], C, norm="ln"), Xa, ya, Xv, yv, C, seed); rows.append(dict(setting="Centralized", partition="-", seed=seed, **scores(yt, logits(net, Xt).argmax(1))))
    for alpha in [None] + CFG["DIR_ALPHAS"]:
        pname = "IID" if alpha is None else f"Dirichlet α={alpha}"
        parts_ = partition(tr, CFG["CLIENTS"], alpha, seed)
        Xs = [sc(XALL[p]) for p in parts_]; ys = [d.fam.map(cl).values[p] for p in parts_]
        loc = [scores(yt, logits(train(MLP(Xa.shape[1], C, norm="ln"), X, y, Xv, yv, C, seed, epochs=min(CFG["EPOCHS"], 30)), Xt).argmax(1)) for X, y in zip(Xs, ys) if len(np.unique(y)) > 1]
        rows.append(dict(setting="Local-only (mean)", partition=pname, seed=seed, acc=np.mean([l["acc"] for l in loc]), mf1=np.mean([l["mf1"] for l in loc])))
        for name, mu in [("FedAvg", 0.0), ("FedProx", CFG["MU"])]:
            g, hist, _ = fed_run(Xs, ys, Xv, yv, Xt, yt, C, CFG["ROUNDS"], mu, None, seed, Xa.shape[1])
            rows.append(dict(setting=name, partition=pname, seed=seed, **scores(yt, logits(g, Xt).argmax(1)))); curves.setdefault((name, pname), []).append(hist)
        print("E3", seed, pname, flush=True)
E3 = pd.DataFrame(rows); E3.to_csv(f"{CFG['OUT']}/E3_federated_raw.csv", index=False)
E3s = E3.groupby(["setting", "partition"])[["acc", "mf1"]].agg(["mean", "std"]).round(4); E3s.to_csv(f"{CFG['OUT']}/E3_federated.csv"); print(E3s)
for (s_, p_), g in E3.groupby(["setting", "partition"]): R[f"E3_{s_}_{p_}_MF1".replace(" ", "")] = fmt(g.mf1)
save()

## E4 — differential privacy (DP-SGD inside FedAvg, Dirichlet α=0.5)

In [ ]:
rows = []
for seed in CFG["SEEDS"]:
    tr, va, te = split("grouped", seed); Xa, ya, Xv, yv, Xt, yt, cls = arrays("fam", tr, va, te); C = len(cls)
    sc = Scaler().fit(XALL[tr]); cl = {c: i for i, c in enumerate(cls)}
    parts_ = partition(tr, CFG["CLIENTS"], CFG["DIR_ALPHAS"][0], seed); Xs = [sc(XALL[p]) for p in parts_]; ys = [d.fam.map(cl).values[p] for p in parts_]
    for sigma in CFG["DP_SIGMAS"]:
        g, _, steps = fed_run(Xs, ys, Xv, yv, Xt, yt, C, CFG["ROUNDS"], 0.0, sigma, seed, Xa.shape[1], track=False)
        eps = float("nan")
        if RDPAccountant is not None:
            nmin = min(len(x) for x in Xs if len(x)); acc_ = RDPAccountant()
            acc_.history = [(sigma, min(1.0, 128 / nmin), steps * CFG["ROUNDS"])]; eps = acc_.get_epsilon(CFG["DP_DELTA"])
        rows.append(dict(sigma=sigma, epsilon=eps, seed=seed, **scores(yt, logits(g, Xt).argmax(1)))); print("E4", seed, sigma, round(eps, 2), flush=True)
E4 = pd.DataFrame(rows); E4.to_csv(f"{CFG['OUT']}/E4_privacy_raw.csv", index=False)
E4s = E4.groupby("sigma")[["epsilon", "acc", "mf1"]].agg(["mean", "std"]).round(4); E4s.to_csv(f"{CFG['OUT']}/E4_privacy.csv"); print(E4s)
for s_, g in E4.groupby("sigma"): R[f"E4_sigma{s_}_EPS"] = round(float(g.epsilon.mean()), 2); R[f"E4_sigma{s_}_MF1"] = fmt(g.mf1)
save()

## E5 — unseen-family recognition (leave-one-family-out)

In [ ]:
def knn_s(A, B, k):
    n = lambda z: z / (np.linalg.norm(z, axis=1, keepdims=True) + 1e-12); return NearestNeighbors(n_neighbors=k).fit(n(A)).kneighbors(n(B))[0][:, -1]
def maha_s(A, ya, B):
    us = np.unique(ya); mus = np.stack([A[ya == c].mean(0) for c in us]); P = np.linalg.pinv(np.cov((A - mus[np.searchsorted(us, ya)]).T) + 1e-6 * np.eye(A.shape[1]))
    return np.stack([np.einsum("ij,jk,ik->i", B - m, P, B - m) for m in mus], 1).min(1)
@torch.no_grad()
def body(net, X): return torch.cat([net.body(T(X[k:k + 8192])) for k in range(0, len(X), 8192)]).cpu().numpy()
def fitT(lg, y):
    best = (1e9, 1.0)
    for t in np.logspace(-1.5, 1.5, 61):
        p = sm(lg / t); nll = -np.log(p[np.arange(len(y)), y] + 1e-12).mean(); best = min(best, (nll, t))
    return best[1]

rows = []
tr, va, te = split("grouped", 0)
for hold in FAMS:
    cls = ["Benign"] + [f for f in FAMS if f != hold]; cl = {c: i for i, c in enumerate(cls)}; C = len(cls)
    keep = lambda ix: ix[d.fam.values[ix] != hold]; a, v = keep(tr), keep(va)
    sc = Scaler().fit(XALL[a]); Xa, Xv, Xt = sc(XALL[a]), sc(XALL[v]), sc(XALL[te])
    fmap = d.fam.map(cl).values; ya, yv = fmap[a].astype(int), fmap[v].astype(int); unk = d.fam.values[te] == hold
    for seed in CFG["SEEDS"]:
        S = {}; net = train(MLP(Xa.shape[1], C), Xa, ya, Xv, yv, C, seed); lg = logits(net, Xt); p = sm(lg); tau = fitT(logits(net, Xv), yv)
        S["MSP"] = 1 - p.max(1); S["MSP+TS"] = 1 - sm(lg / tau).max(1)
        S["Energy"] = -tau * np.log(np.exp((lg - lg.max(1, keepdims=True)) / tau).sum(1)) - lg.max(1)
        pm = np.stack([sm(logits(net, Xt, mc=True)) for _ in range(CFG["MC_T"])]); S["MC-Dropout MI"] = ent(pm.mean(0)) - np.mean([ent(q) for q in pm], 0)
        Za, Zt = body(net, Xa), body(net, Xt); S["kNN (embedding)"] = knn_s(Za, Zt, CFG["KNN_K"]); S["Mahalanobis (embedding)"] = maha_s(Za, ya, Zt)
        e = train(MLP(Xa.shape[1], C), Xa, ya, Xv, yv, C, seed, kind="edl"); al = Fn.softplus(T(logits(e, Xt))).cpu().numpy() + 1; S["EDL vacuity"] = C / al.sum(1)
        o = train(MLP(Xa.shape[1], C), Xa, ya, Xv, yv, C, seed, kind="ovr"); S["One-vs-rest (DOC)"] = 1 - (1 / (1 + np.exp(-logits(o, Xt)))).max(1)
        for k_, s_ in S.items(): rows.append(dict(family=hold, method=k_, seed=seed, auroc=roc_auc_score(unk, s_)))
        rows.append(dict(family=hold, method="_flagged_as_malware", seed=seed, auroc=float((p.argmax(1)[unk] != 0).mean())))
    parts_ = partition(a, CFG["CLIENTS"], CFG["DIR_ALPHAS"][0], 0)
    g, _, _ = fed_run([sc(XALL[q]) for q in parts_], [fmap[q].astype(int) for q in parts_], Xv, yv, Xt, None, C, CFG["LOFO_ROUNDS"], 0.0, None, 0, Xa.shape[1], track=False)
    rows.append(dict(family=hold, method="FedAvg MSP", seed=0, auroc=roc_auc_score(unk, 1 - sm(logits(g, Xt)).max(1))))
    print("E5", hold, flush=True)
E5 = pd.DataFrame(rows); E5.to_csv(f"{CFG['OUT']}/E5_unseen_raw.csv", index=False)
E5p = E5[~E5.method.str.startswith("_")].pivot_table(index="method", columns="family", values="auroc", aggfunc="mean").round(3)
E5p["MEAN"] = E5p.mean(1).round(3); E5p = E5p.sort_values("MEAN", ascending=False); E5p.to_csv(f"{CFG['OUT']}/E5_unseen_auroc.csv"); print(E5p)
for m_, v_ in E5p.MEAN.items(): R[f"E5_{m_}_AUROC"] = float(v_)
R["E5_unknown_flagged_as_malware"] = round(float(E5[E5.method == "_flagged_as_malware"].auroc.mean()), 4)
save()

## Figures

In [ ]:
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False, "savefig.dpi": 300, "axes.grid": True, "grid.alpha": .25})
BLUE, ORANGE, GREY = "#0072B2", "#E69F00", "#999999"
# Fig 1: leakage
fig, axs = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, task, title in zip(axs, ["bin", "cat", "fam"], ["Binary", "Category (4)", "Family (16)"]):
    g = E1[E1.task == task].groupby(["model", "split"]).mf1.agg(["mean", "std"]).reset_index(); ms = ["RF", "XGBoost", "MLP"]; x = np.arange(len(ms))
    for o, sp, c in [(-.18, "random", GREY), (.18, "grouped", BLUE)]:
        gg = g[g.split == sp].set_index("model").loc[ms]; ax.bar(x + o, gg["mean"], .34, yerr=gg["std"], color=c, label="random record split" if sp == "random" else "grouped by sample")
    ax.set_xticks(x); ax.set_xticklabels(ms); ax.set_title(title); ax.set_ylabel("Macro-F1"); ax.set_ylim(0, 1.05)
axs[0].legend(fontsize=8, loc="lower left"); fig.tight_layout(); fig.savefig(f"{CFG['OUT']}/fig_leakage.png"); plt.close(fig)
# Fig 2: federated curves
fig, ax = plt.subplots(figsize=(7, 4)); cols = {"IID": BLUE, "Dirichlet α=0.5": ORANGE, "Dirichlet α=0.1": "#D55E00"}
for (name, pname), hs in curves.items():
    r_ = [h[0] for h in hs[0]]; m_ = np.mean([[v for _, v in h] for h in hs], 0)
    ax.plot(r_, m_, "-" if name == "FedAvg" else "--", color=cols.get(pname, GREY), label=f"{name} · {pname}")
cen = E3[E3.setting == "Centralized"].mf1.mean(); ax.axhline(cen, color="black", lw=1, ls=":", label=f"Centralized ({cen:.3f})")
ax.set(xlabel="Communication round", ylabel="Family macro-F1 (test)"); ax.legend(fontsize=7.5); fig.tight_layout(); fig.savefig(f"{CFG['OUT']}/fig_federated.png"); plt.close(fig)
# Fig 3: privacy-utility
if len(E4):
    g = E4.groupby("sigma").agg(eps=("epsilon", "mean"), m=("mf1", "mean"), s=("mf1", "std")).reset_index()
    fig, ax = plt.subplots(figsize=(5.5, 3.8)); xs = g.eps if g.eps.notna().all() else g.sigma
    ax.errorbar(xs, g.m, yerr=g.s, fmt="-o", color=BLUE); ax.axhline(E3[(E3.setting == "FedAvg") & (E3.partition == f"Dirichlet α={CFG['DIR_ALPHAS'][0]}")].mf1.mean(), color=GREY, ls="--", label="FedAvg without DP")
    ax.set(xlabel="ε (δ=1e-5)" if g.eps.notna().all() else "noise multiplier σ", ylabel="Family macro-F1"); ax.legend(fontsize=8); fig.tight_layout(); fig.savefig(f"{CFG['OUT']}/fig_privacy.png"); plt.close(fig)
# Fig 4: unseen-family heatmap (skipped if E5 did not finish)
if 'E5p' in globals():
    fig, ax = plt.subplots(figsize=(12, 4.2)); M = E5p.drop(columns="MEAN")
    im = ax.imshow(M.values, cmap="Blues", vmin=0.3, vmax=1.0, aspect="auto")
    ax.set_xticks(range(M.shape[1])); ax.set_xticklabels(M.columns, rotation=40, ha="right", fontsize=8); ax.set_yticks(range(M.shape[0])); ax.set_yticklabels([f"{i} ({E5p.MEAN[i]:.3f})" for i in M.index], fontsize=8)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]): ax.text(j, i, f"{M.values[i, j]:.2f}", ha="center", va="center", fontsize=6.5, color="white" if M.values[i, j] > .75 else "black")
    ax.grid(False); fig.colorbar(im, ax=ax, fraction=.02, label="AUROC (held-out family vs rest)"); ax.set_title("Unseen-family recognition (leave-one-family-out)")
    fig.tight_layout(); fig.savefig(f"{CFG['OUT']}/fig_unseen_family.png"); plt.close(fig)
# Fig 5: family confusion matrix (best of RF / FT-Transformer / MLP by macro-F1)
best = E2[E2.task == "fam"].groupby("model").mf1.mean().idxmax(); best = best if best in cms else "RF"; cm, cls = cms[best]
cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1); fig, ax = plt.subplots(figsize=(8, 7)); ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(cls))); ax.set_xticklabels(cls, rotation=60, ha="right", fontsize=8); ax.set_yticks(range(len(cls))); ax.set_yticklabels(cls, fontsize=8)
for i in range(len(cls)):
    for j in range(len(cls)):
        if cmn[i, j] >= .05: ax.text(j, i, f"{cmn[i,j]:.2f}", ha="center", va="center", fontsize=6, color="white" if cmn[i, j] > .6 else "black")
ax.grid(False); ax.set(xlabel="Predicted", ylabel="True", title=f"Family confusion, grouped split ({best})"); fig.tight_layout(); fig.savefig(f"{CFG['OUT']}/fig_family_confusion.png"); plt.close(fig)
R["GPU"] = torch.cuda.get_device_name(0) if DEV.type == "cuda" else "CPU"; save()
print("DONE ->", CFG["OUT"]); print(sorted(os.listdir(CFG["OUT"])))